In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [4]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Q1 - Ans - 490+324 = 814

In [5]:
train_copy = train.copy()

In [7]:
import string
train_copy['prompt2'] = train_copy['prompt'].apply(lambda x:"".join(ch for ch in x if ch not in string.punctuation))
cont = "".join(s for s in train_copy['prompt2'])
s = cont.split(" ")    
st = set()
for i in s:
    st.add(i.lower())
print(len(st))

Q2 - ans = 1318

In [18]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

words = train_copy['prompt2'].iloc[0].lower().split()

filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]

result = " ".join(filtered_words)

print(result)
print(len(filtered_words))

pick best possible answer martin heideggers view relationship time human existence listed options
13


Q3 - Ans= 13

In [19]:
train_copy.columns

Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'prompt2'], dtype='object')

In [20]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

documents = (
    train_copy["prompt"].fillna("") + " " +
    train_copy["A"].fillna("") + " " +
    train_copy["B"].fillna("") + " " +
    train_copy["C"].fillna("") + " " +
     train_copy["D"].fillna("") + " " + train_copy["E"]
).tolist()

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(documents)

vocab_size = len(vectorizer.vocabulary_)

print(vocab_size)

2762


Q3 - Ans=2762

In [26]:
from sklearn.metrics.pairwise import cosine_similarity
prompt_vec = vectorizer.transform([train_copy['prompt'].iloc[0]])
optionA_vec = vectorizer.transform([train_copy['A'].iloc[0]])

sim = cosine_similarity(prompt_vec, optionA_vec)[0][0]

print(round(sim, 4))

0.272


Q4 - Ans= 0.272

In [29]:
arr = np.zeros(5)
for i in range(5):
    arr[i] = i
print(np.argmax(arr))

4


In [30]:
cols = ["A","B","C","D","E"]
t =0
for i in range(train_copy.shape[0]):
    prompt_vec = vectorizer.transform([train_copy['prompt'].iloc[i]])
    sims = np.zeros(5)
    for j in range(5):
        option_vec = vectorizer.transform([train_copy[cols[j]].iloc[i]])
        sim = cosine_similarity(prompt_vec,option_vec)[0][0]
        sims[j] = sim
    idx = np.argmax(sims)
    if cols[idx] == train_copy["answer"].iloc[i]:
        t += 1 
print(t/train_copy.shape[0])
    
    
    

0.1355


Q5 - Ans = 13.55

Q6 - Ans = 1/1 = 1.0

In [33]:
train_copy['answer'].value_counts()


answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [36]:
sm = 0.0

for i in range(train_copy.shape[0]):
    corr = train_copy["answer"].iloc[i]
    if  corr == "B":
        sm += 1.0
    elif corr == "C":
        sm += 0.5 
    elif corr == "A":
        sm += 1/3
    
print(sm/train_copy.shape[0])

0.4212500000000017


Q7 - Ans = 0.4213

In [38]:
cols = ["A","B","C","D","E"]
sm =0.0
for i in range(train_copy.shape[0]):
    prompt_vec = vectorizer.transform([train_copy['prompt'].iloc[i]])
    sims = np.zeros(5)
    for j in range(5):
        option_vec = vectorizer.transform([train_copy[cols[j]].iloc[i]])
        sim = cosine_similarity(prompt_vec,option_vec)[0][0]
        sims[j] = sim
    sorted_idx = np.argsort(sims)[::-1]
    corr = train_copy["answer"].iloc[i]
    if  corr == cols[sorted_idx[0]]:
        sm += 1.0
    elif corr == cols[sorted_idx[1]]:
        sm += 0.5 
    elif corr == cols[sorted_idx[2]]:
        sm += 1/3
    
    
print(sm/train_copy.shape[0])

0.2552499999999985


Q7- Ans  = 0.2552